# **Project Overview**
#### First we are going to have a Planner AI Agent, which is going to be powered by OpenAI API. We are going to integrate politics guardrails inside that planner AI agent.
#### So basically if the user provides a topic, for example, to that planner agent, and that topic has anything related to politics, we are just going to tigger guardrail and exit the process.
#### After this, we are going to hand off the tasks from the planner agent to another AI agent called "Writer AI Agent", which is also going to be powered by OpenAI API.
#### Basically what we are doing here is how can an AI agent take a task and just give it to another AI agent.
#### The next thing we are going to do is to make sure that "Writer AI Agent" has access to tools. And these tools are going to be another AI agents. These tools include:
#### 1. Tavily Search Agent (Tool): This agent is going to be wrapped as a tool and made available to the "Writer AI Agent" to use it whenever it wants. A writer AI agent is going to ask the "Tavily Search Agent" that "Please give me information about these 2-3 search queries. And the Tavily Search Agent will provide research results or search results. The only trick here is how we can get an AI agent, wrap it as a tool and make it available to another AI agent.
#### 2. Fundamental AI Agent: You can use this project a little bit in finance. You want a "Writer AI Agent" to write a full report about a startup in the electric vehicle battery space  and what that writer AI agent needs to do. It needs to go and search the web (Tavily Search Agent). It is also going to perform fundamental analysis, to know or take a look at the revenues of the company, all the debts, if dividends are being paid to the shareholders and so on.
#### So what the "Writer AI Agent" is going to do to leverage or to get any fundamental analysis from "Fundamental AI Agent" (tool) and research or search results from "Tavily Search AI Agent" (tool). Once the "Writer AI Agent" is done with the task, it is going to generate a full report at the end that includes fundamental analysis, and also research results about a given company.
#### So, the "Writer AI Agent" is going to work autonomously. If it needs fundamental analysis tool or search tool depends on user's query.
### Key Learning Objectives:
#### 1. Design a multi-agent research assistant that can triage user's queries, delegate tasks to specialist agents, and generate an executive-ready report.
#### 2. Develop a library of specialist agents, (e.g,Planner, Writer, Fundamental Analyst, Tavily Search Agent) and coordinate their interactions.
#### 3. Implement guardrails to enforce boundaries, such as preventing responses on restricted topics (e.g, Politics).
#### 4. Build handoff mechanisms that transfer context and imputs smoothlybetween agents (e.g, Planner-Writer).
#### 5. Create a tool as an agent by wrapping an autonomous agent behind a function-tool interface, allowing it to be invoked seamlessly by other agents.
#### 6. Apply memory orchestration  (via sessions and runners) so that agents can track past steps and adapt to reasoning dynamically.

In [ ]:
!pip install -q --upgrade openai-agents==0.2.2

In [ ]:
!pip install helper

In [ ]:
import os, requests, asyncio
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from agents import Agent, Runner, function_tool, SQLiteSession
from pydantic import BaseModel
from typing_extensions import TypedDict
from datetime import datetime
from agents import RunResult
from agents import handoff, RunContextWrapper, CodeInterpreterTool, InputGuardrail, GuardrailFunctionOutput, TResponseInputItem
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

load_dotenv()

from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

tavily_api_key = userdata.get("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = tavily_api_key

print(f"Retrieved API Key: {openai_api_key[:5]}...{openai_api_key[-5:]}" if openai_api_key else "OpenAI API Key not found or is None")
openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI Client successfully configured.")

Retrieved API Key: sk-pr...c_bMA
OpenAI Client successfully configured.


In [ ]:
!pip install -q tavily-python
from tavily import TavilyClient

# Configure the Tavily client separately
tavily_client = TavilyClient(api_key=tavily_api_key)
print("Tavily Client successfully configured.")

Tavily Client successfully configured.


In [ ]:
def print_markdown(txt: str):
    display(Markdown(txt))

### **Define the models to be used**

In [ ]:
main_model = "gpt-4.1-mini"
small_model = "gpt-4.1-nano"

### **Define the Tavily Search Function**

In [ ]:
class TavilyParams(TypedDict):
    query: str
    max_results: int

@function_tool
def tavily_search(params: TavilyParams) -> str:
    """Return a newline-joined summary of Tavily results."""
    url = "https://api.tavily.com/search"

    payload = {
        "api_key": os.environ.get("TAVILY_API_KEY", ""),
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }

    resp = requests.post(url, json=payload, headers={"Content-Type": "application/json"})
    if resp.status_code != 200:
        return f"Tavily error {resp.status_code}"

    items = resp.json().get("results", [])
    return "\n".join([f".{itm['title']}: {itm['content']}" for itm in items]) or "No hits"

### **Let's establish a political Guardrail here.**

In [ ]:
import os
from pydantic import BaseModel
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem

# Define the output schema for the political topic guardrail
class PoliticalTopicOutput(BaseModel):
    is_political: bool
    reasoning: str

# Define the guardrail classification agent
politics_guardrail_agent = Agent(
    name="Guardrail Check",
    instructions="Check if the user is asking about political topics, politicians, elections, government, and so on",
    output_type=PoliticalTopicOutput
)

# Corrected guardrail function to use Runner.run instead of Runner.run_async
async def politics_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    # Use the standard Runner.run method which is already an async coroutine
    result = await Runner.run(politics_guardrail_agent, input=user_message)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_political
    )

# Instantiate the InputGuardrail
politics_guardrail = InputGuardrail(politics_guardrail_fn)

## **We make "Politic_Guardrail_Agent" available to "Planner AI Agent" here.**
### **Now let's define a Planner AI Agent**

In [ ]:
from datetime import datetime
from pydantic import BaseModel
from agents import Agent

# Ensure model configuration is defined
main_model = "gpt-4o-mini"  # Using standard gpt-4o-mini model

# Define a data model for one search plan item where each item contains:
# 1. reason: Why this particular search is being done.
# 2. query: The actual search string to run.

class SearchPlanItem(BaseModel):
    reason: str
    query: str

# Define a data model for the complete search plan.
# This contains a list of SearchPlanItems.

class SearchPlan(BaseModel):
    searches: list[SearchPlanItem]

# Get today's data in YYYY-MM-DD format.
date = datetime.now().strftime("%Y-%m-%d")

# Create an AI agent called "Planner"
planner_agent = Agent(name = "Planner",
                      instructions = f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive
        research plan for a user request. You have access to web search tools and should utilize the current date ({date})) when planning.
        Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.
        Ensure coverage of recent news, company fundamentals, risks, sentiments, and broader context.

        Input: The user's research request and the current date.

        Output: A list of search plan items , each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.
        """,
            model = main_model,
            output_type = SearchPlan,
            input_guardrails = [politics_guardrail])  # This is the guardrail that prevents political topics being discussed.

### **Now let's create and test a "Planner AI Agent" and test it with the "Guardrail" without a political question**

In [ ]:
import os
from google.colab import userdata
from IPython.display import display, Markdown
from agents import Runner, Agent, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem
from pydantic import BaseModel

# Ensure the API Key is fetched directly and set in the environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

# 1. Force-update the guardrail function and variable in the active kernel memory
async def politics_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    # Use the correct Runner.run method
    result = await Runner.run(politics_guardrail_agent, input=user_message)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_political
    )

politics_guardrail = InputGuardrail(politics_guardrail_fn)

# 2. Create the planner agent to use the updated guardrail instance
planner_agent = Agent(
    name="Planner",
    instructions=f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive\n        research plan for a user request. You have access to web search tools and should utilize the current date ({date})) when planning.\n        Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.\n        Ensure coverage of recent news, company fundamentals, risks, sentiments, and broader context.\n\n        Input: The user's research request and the current date.\n\n        Output: A list of search plan items , each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.\n        """,
    model=main_model,
    output_type=SearchPlan,
    input_guardrails=[politics_guardrail]
)

# 3. Execute the test run with the updated code
question1 = "Solid state battery companies"
run1 = await Runner.run(starting_agent=planner_agent, input=question1)
display(Markdown(f"### Agent's Answer\n{run1.final_output}"))

### Agent's Answer
searches=[SearchPlanItem(reason='To gather the latest developments and news regarding solid-state battery technology and key players in the industry.', query='latest news solid state battery companies October 2026'), SearchPlanItem(reason='To analyze the financial performance and business fundamentals of leading companies developing solid-state batteries, which will provide insights into their market position and viability.', query='solid state battery companies financial analysis 2026'), SearchPlanItem(reason='To assess potential risks and market sentiment surrounding solid-state batteries, including challenges in production, competition, and regulatory issues impacting the industry.', query='risks and sentiments solid state battery market 2026')]

### **Let's creare and test a "Planner AI Agent" with the "Guardrail" with a political question**

In [ ]:
import os
from google.colab import userdata
from IPython.display import display, Markdown
from agents import Runner, Agent, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel

# Ensure the API Key is fetched directly and set in the environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

# 1. Force-update the guardrail function and variable in the active kernel memory
async def politics_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    # Use the correct Runner.run method
    result = await Runner.run(politics_guardrail_agent, input=user_message)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_political
    )

politics_guardrail = InputGuardrail(politics_guardrail_fn)

# 2. Re-bind the planner agent to use the updated guardrail instance
planner_agent = Agent(
    name="Planner",
    instructions=f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive\n        research plan for a user request. You have access to web search tools and should utilize the current date ({date})) when planning.\n        Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.\n        Ensure coverage of recent news, company fundamentals, risks, sentiments, and broader context.\n\n        Input: The user's research request and the current date.\n\n        Output: A list of search plan items , each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.\n        """,
    model=main_model,
    output_type=SearchPlan,
    input_guardrails=[politics_guardrail]
)

# 3. Execute the test run with the updated code and catch the guardrail exception
question1 = "Why is Trump meeting with Putin this week?"
try:
    run1 = await Runner.run(starting_agent=planner_agent, input=question1)
    display(Markdown(f"### Agent's Answer\n{run1.final_output}"))
except InputGuardrailTripwireTriggered as e:
    display(Markdown("### Guardrail Triggered 🚫\nYour request was blocked because it contains political content."))

### Guardrail Triggered 🚫
Your request was blocked because it contains political content.

## **Practice Opportunity**
### Create a new guardrail that will block the agent from discussing topics related to defence and military.
### -Update the guardrail agent's instructions so that it focusses on analyzing defence and military-related issues and risk.
### -Instantiate the defence_guardrail agent.
### -Provide an example input (e.g, "Lockheed Martin drones" and test the agent.

In [ ]:
import os
from pydantic import BaseModel
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem

# Define the output schema for the defense topic guardrail
class DefenseTopicOutput(BaseModel):
    is_defense: bool
    reasoning: str

# Define the guardrail classification agent
defense_guardrail_agent = Agent(
    name="Guardrail Check",
    instructions="",
    output_type=DefenseTopicOutput
)

# Corrected guardrail function to use Runner.run instead of Runner.run_async
async def defense_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    # Use the standard Runner.run method which is already an async coroutine
    result = await Runner.run(defense_guardrail_agent, input=user_message)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_defense
    )

# Instantiate the InputGuardrail
defense_guardrail = InputGuardrail(defense_guardrail_fn)

### Let's create "Planner AI Agent" and test it with the "Guardrail" with a defense question

In [ ]:
import os
from google.colab import userdata
from IPython.display import display, Markdown
from agents import Runner, Agent, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel

# Ensure the API Key is fetched directly and set in the environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key

# 1. Force-update the guardrail function and variable in the active kernel memory
async def defense_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    # Use the correct Runner.run method
    result = await Runner.run(defense_guardrail_agent, input=user_message)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_defense
    )

defense_guardrail = InputGuardrail(defense_guardrail_fn)

# 2. Re-bind the planner agent to use the updated guardrail instance
planner_agent = Agent(
    name="Planner",
    instructions=f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive\n
            research plan for a user request. You have access to web search tools and should utilize the current date ({date})) when planning.\n
            Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.\n
            Ensure coverage of recent news, company fundamentals, risks, sentiments, and broader context.\n\n
            Input: The user's research request and the current date.\n\n
            Output: A list of search plan items , each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.
            """,
    model=main_model,
    output_type=SearchPlan,
    input_guardrails=[defense_guardrail]
)

# 3. Execute the test run with the updated code and catch the guardrail exception
question1 = "Lockheed martin drones"
try:
    run1 = await Runner.run(starting_agent=planner_agent, input=question1)
    display(Markdown(f"### Agent's Answer\n{run1.final_output}"))
except InputGuardrailTripwireTriggered as e:
    display(Markdown("### Guardrail Triggered 🚫\nYour request was blocked because it contains defense content."))

### Guardrail Triggered 🚫
Your request was blocked because it contains defense content.

### **Define a Team of AI Agents: Tavily Search AI Agent and Fundamental Analysis Agent.**

### **Tavily AI Agent created as a tool**

In [ ]:
# Let's define the Tavily Search AI Agent, which finds and summaries the most recent, relevant information for a search query.
class Summary(BaseModel):
    summary: str

search_agent = Agent(name = "Searcher",
                     instructions = """

Context: You are a search specialist agent with access to the Tavily web search tool.
Your goal is to provide up-to-date, relevant information for a research task.

Instruction: Use Tavily search to find the most relevant and pertinent information
related to the user's query. Summarize your findings clearly and concisely in no more than 200 words.

Input: The user's search query.

# Output: A concise summary (<=200 words) of the most relevant and recent information found via Tavily search.""",

      tools = [tavily_search],
      model = main_model,
      output_type = Summary)

### **Fundamental Analysis AI Agent created as a tool**

In [ ]:
# Let's define the Fundamental Analysis AI Agent.
class Summary(BaseModel):
    summary: str

fundamentals_agent = Agent(name = "FundamentalAnalyst",
                     instructions = """

Context: You are a financial analyst specializing in company fundamentals.

Instruction: Carefully analyze the provided notes to assess the company's financial fundamentals, including revenue, growth, and so on.

Input: Notes containing relevant financial data and qualitative information about the company

# Output: A concise summary (<=200 words) highlighting key points about the company'srevenue, growth trajectory, and profit margins.

Tools: The following are available for comprehensive research on the company:
-tavily_search: Search the web for information about the company.
""",
tools = [tavily_search],
model = main_model,
output_type = Summary)

### **Create Agents as Tools for the Writer AI Agent.**

In [ ]:
# Let's define a writer agent which synthesizes Search and Analysts insights into a cohesive Investment Report.
# Define the output format for the writer agent using pydantic as BaseModel.
# This ensures the agent always return:
# 1. Short_summary: a brief 2-3 sentence executive summary.
# 2. Markdown_report: a detailed markdown formatted report.
# 3. Follow-up-questions: 3-5 suggested research questions.

class FinalReport(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_questions: list[str]

# Define an asynchronous helper function to extract the "Summary" field from the final_output of another agent's run result.
# This is used so the writer agent can easily consume outputs of other tools.

async def extract_summary(run_result: RunResult) -> str:
    """Extract the 'summary' field from the final_output of an agent run"""
    return run_result.final_output.summary

# Note that fundamentals_agent and search_agent are other AI agents we defined earlier.
# The method .as_tool() wraps each of them into a callable tool so that the writer_agent can make use of them.

tools = [
    fundamentals_agent.as_tool("fundamentals",
                               "Get fundamental analysis",
                               custom_output_extractor = extract_summary,  # Use the async function here.
),
  search_agent.as_tool(
      "search",
      "Get search results",
      custom_output_extractor = extract_summary,  # Use the async function here.
  ),
]

# Define the writer agent.
writer_agent = Agent(name = "Writer",
                     instructions = """

Context: You are an expert research writer preparing a comprehensive investment report on a company, using diverse information sources.
Your audience is sophisticated and expects clarity, depth, and actionable insihts.

Instruction: Thoroughly analyze the provided search snippets and analyst summaries. Synthesize these into a cohesive, well-structured
markdown report of at least 600 words. Your report must:
1. begin with a concise 2-3 sentence executive summary capturing the most critical findings;
2. integrate and cross-reference key facts, trends and information from all sources, highlighting both concensus and disagreements;
3. organize content with clear headings and logical flow;
4. maintain detailed instructions.
Ensure the writing is precise, professional, and tailored for an investment decision-making context.
You must always use the search tool to gather and incorporate up-to-date information in your report.
The other tools-'functionals' are optional and should only be used if the user specifically requests those analyses or if you
determine that including them would significantly enhance the report's quality or relevance.

Input: A set of search snippets and analyst summaries containing relevant information about the company, and any user instructions
specifying which analyses to include.

Output: A markdown-formatted report (minimum 600 words) including an executive summary and 3-5 well crafted follow-up research questions.

Tools: The following are available for comprehensive research on the company:
-fundamentals: Get fundamentals analysis (optional).
-search: Get search results (required).
""",
model = main_model,
output_type = FinalReport,
tools = tools
)

### **Now let's test the "Writer AI Agent"**

In [ ]:
question1 = "Do a deep dive on the latest news in Tesla stock. I also need the fundamental analysis of the company."
run1 = await Runner.run(starting_agent = writer_agent, input = question1)
print_markdown(f"### Agent's Answer\n{run1.final_output}")

### Agent's Answer
# Tesla Investment Report

## Executive Summary
Tesla Inc. continues to captivate the stock market with its dynamic developments and strong fundamental performance. Recent reports highlight Tesla's strategic advancements in global production capacity and innovative ventures into energy solutions. Conversely, market analysts express mixed views on Tesla’s aggressive expansion strategy, which has implications for its valuation metrics. Both opportunities and challenges merit attention for informed investment decisions.

## Recent Developments

### Expansion in Production and Technological Innovation
Tesla has recently announced the expansion of its production facilities, notably in Texas and Germany. This expansion aligns with Tesla's strategy to increase its global vehicle output, meeting rising demand and leveraging economies of scale. Additionally, Tesla's continuous advancements in battery technology signify its commitment to maintaining a competitive edge in the electric vehicle (EV) market.

### Energy Solutions and Sustainability
Tesla is not just a car company; its venture into energy solutions through its solar products and energy storage systems positions it as a major player in sustainability. The company’s focus on reducing cost per kilowatt-hour for its battery solutions underlines a strategic move to capture a larger share of the renewable energy market.

## Financial Performance and Fundamental Analysis

### Revenue Growth and Profitability
Tesla has reported significant revenue growth over the past year, highlighting robust vehicle sales and increased deliveries. However, profit margins have been under pressure due to higher raw material costs and investments in new production facilities. The company's gross margin remains higher than industry averages, reflecting efficiency in production and robust pricing power.

### Valuation and Market Sentiment
While Tesla’s price-to-earnings (P/E) ratio is significantly higher than traditional automakers, reflecting strong growth prospects and investor optimism, some analysts express concerns about overvaluation. These concerns stem from potential risks associated with scaling production and navigating regulatory environments in new markets.

### Balance Sheet and Cash Flow
Tesla maintains a strong balance sheet, characterized by a healthy cash position and manageable levels of debt. Its free cash flow has been consistently positive, supporting ongoing investments in R&D and facility expansions. Analysts appreciate Tesla’s financial discipline, which underpins its growth initiatives.

## Market Position and Competition

### Dominance in the EV Market
Tesla’s leadership in the EV market is undeniable, buoyed by its strong brand loyalty and innovative product pipeline. However, increasing competition from established automakers entering the EV space could challenge Tesla’s market share.

### Strategic Partnerships and Collaborations
Investments in partnerships and collaborations have been pivotal in Tesla's strategy to enhance its technological capabilities and market reach. For instance, collaborations with lithium suppliers aim to secure long-term supply chains for critical raw materials.

## Analyst Opinions and Diverging Perspectives

### Bullish Outlook
Proponents of Tesla stock emphasize the company’s transformative impact on the automotive industry and its potential to disrupt energy markets. They argue that Tesla’s technological advancements and ambitious expansion goals justify its stretched valuations.

### Bearish Concerns
Skeptical analysts point out the risks involved with rapid expansion, such as operational inefficiencies and the potential for regulatory scrutiny in new territories. They caution that such factors could pressure Tesla’s margins and heighten investor risk if growth expectations falter.

## Follow-Up Research Questions

1. How will Tesla's expansion in Texas and Germany affect global EV market dynamics and projected delivery numbers?
2. What impact will increasing competition in the EV market have on Tesla's pricing strategies and market share?
3. How are Tesla's energy solutions being received in international markets, and what growth potential exists in those sectors?
4. To what extent do potential regulatory changes in key markets pose a risk to Tesla's operational strategies?
5. How might Tesla's partnerships in raw materials affect long-term sustainability and cost optimization?

In summary, Tesla remains a formidable player with a diversified portfolio extending beyond vehicles into energy solutions. While expansion presents significant opportunities, the inherent risks necessitate a balanced assessment for potential investors. As Tesla navigates its dynamic growth path, staying abreast of its strategic, financial, and operational developments is crucial for informed investment decisions.

## **AI Agents Handoff**

In [ ]:
# Create a session to store the handoff history.
session = SQLiteSession("research_agent_handoff")

In [ ]:
# Define the input schema for transfering from planner to writer.
class PlannerToWriterInput(BaseModel):
    original_query: str    # The original user query string
    search_plan: SearchPlan  # The search plan generated by the planner.

# Callback function that runs when planner hands off to writer.
def on_planner_to_writer(ctx: RunContextWrapper[None], input_data: PlannerToWriterInput):
    print_markdown("Transfer: Planner-Writer")   # Print a message indicating the handoff took place.


# Define the actual handoff setup from Planner to Writer.
handoff_to_writer = handoff(
    agent = writer_agent,  # The target agent (Writer) that will receive the handoff.
    input_type = PlannerToWriterInput,  # The type of input data expected by the writer.
    on_handoff = on_planner_to_writer,  # The callback triggered during handoff.
    tool_name_override = "transfer_to_writer",  # Custom tool name for the handoff.
    tool_description_override = "Transfer to writer with original query and search plan")

# Clone the planner agent and add handoff instructions + handoff tool.
# This ensures the planner knows how to call the writer once the search plan is ready.
planner_with_handoff = planner_agent.clone(
    instructions = f"""{RECOMMENDED_PROMPT_PREFIX}\n\n"""
    + planner_agent.instructions
    + "\nwhen you have produced the SearchPlan, call the handoff tool handoff_to_writer with this JSON input: {{original_query:<the user query>, search_plan: <the SearchPlan JSON}}",
    handoffs = [handoff_to_writer])  # Add the handoff to writer.

In [ ]:
"""
This is the main function that runs the handoff chain end-to-end with a sample query.
"""

async def run_handoffs_demo(user_query: str):
    print_markdown(f"## User Query\n{user_query}")   # Display the user query.

    # Start the chain at the Planner, it will handoff to Writer.
    run_res = await Runner.run(planner_with_handoff, user_query, session = session)
    print_markdown("---")   # Separator for output.

    # Check if the final output is from the verifier and display the result.
    report = run_res.final_output

    print_markdown("---")
    print_markdown(f"### Executive Summary\n{report.short_summary}")
    print_markdown("\n\n------------------\n\n")
    print_markdown(f"### Full Report\n{report.markdown_report}")
    print_markdown("\n\n------------------\n\n")
    return run_res

### **Now, let's test if the "Guardrail" is working here by asking a political question**

In [ ]:
# Re-running the handoff demo since writer_agent has now been corrected to return a structured FinalReport object
handoff_result = await run_handoffs_demo("Can you let me know about the stock holdings of people who are in congress?")

## User Query
Can you let me know about the stock holdings of people who are in congress?

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

### **Let's test again by asking an "Economical Question"**

In [ ]:
# 1. Explicitly redefine clean variables (removing any tuple wrapping)
clean_model = "gpt-4o-mini"
clean_output_type = FinalReport

# 2. Re-create the writer agent ensuring clean arguments are passed
writer_agent = Agent(
    name="Writer",
    instructions="""
Context: You are an expert research writer preparing a comprehensive investment report on a company, using diverse information sources.
Your audience is sophisticated and expects clarity, depth, and actionable insights.

Instruction: Thoroughly analyze the provided search snippets and analyst summaries. Synthesize these into a cohesive, well-structured
markdown report of at least 600 words. Your report must:
1. begin with a concise 2-3 sentence executive summary capturing the most critical findings;
2. integrate and cross-reference key facts, trends and information from all sources, highlighting both consensus and disagreements;
3. organize content with clear headings and logical flow;
4. maintain detailed instructions.
Ensure the writing is precise, professional, and tailored for an investment decision-making context.
You must always use the search tool to gather and incorporate up-to-date information in your report.
The other tools-'fundamentals' are optional and should only be used if the user specifically requests those analyses or if you
determine that including them would significantly enhance the report's quality or relevance.

Input: A set of search snippets and analyst summaries containing relevant information about the company, and any user instructions
specifying which analyses to include.

Output: A markdown-formatted report (minimum 600 words) including an executive summary and 3-5 well crafted follow-up research questions.

Tools: The following are available for comprehensive research on the company:
-fundamentals: Get fundamentals analysis (optional).
-search: Get search results (required).
""",
    model=clean_model,
    output_type=clean_output_type,
    tools=tools
)

# 3. Re-create the handoff with the newly instantiated writer_agent
handoff_to_writer = handoff(
    agent=writer_agent,
    input_type=PlannerToWriterInput,
    on_handoff=on_planner_to_writer,
    tool_name_override="transfer_to_writer",
    tool_description_override="Transfer to writer with original query and search plan"
)

# 4. Re-bind the planner_agent with politics_guardrail
planner_agent = Agent(
    name="Planner",
    instructions=f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive\n
    research plan for a user request. You have access to web search tools and should utilize the current date ({date})) when planning.\n
    Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.\n
    Ensure coverage of recent news, company fundamentals, risks, sentiments, and broader context.\n\n
    Input: The user's research request and the current date.\n\n
    Output: A list of search plan items , each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.\n
    """,
    model=clean_model,
    output_type=SearchPlan,
    input_guardrails=[politics_guardrail]
)

# 5. Clone the planner with the corrected handoff tool
planner_with_handoff = planner_agent.clone(
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}\n\n"""
    + planner_agent.instructions
    + "\nwhen you have produced the SearchPlan, call the handoff tool handoff_to_writer with this JSON input: {{original_query:<the user query>, search_plan: <the SearchPlan JSON}}",
    handoffs=[handoff_to_writer]
)

# 6. Run the demo
handoff_result = await run_handoffs_demo("Do a deep dive on the latest news and developments in the AAPL stock. Also i need the fundamentals analysis of the company")

## User Query
Do a deep dive on the latest news and developments in the AAPL stock. Also i need the fundamentals analysis of the company

Transfer: Planner-Writer

---

---

### Executive Summary
Apple Inc. (AAPL) has shown a solid performance in Q3 2026 with record revenues and positive market sentiment amidst competitive pressures in AI. However, the company's stock faced volatility due to concerns about supply constraints and mixed analyst forecasts regarding future growth.



------------------



### Full Report
# Apple Inc. (AAPL) Stock Analysis Report

## Executive Summary
Apple Inc. (AAPL) has continued to make significant strides in its market performance, reporting a record revenue of $109.4 billion in Q3 FY2026, reflecting a 16% increase year-over-year. Despite this performance, the stock has faced some volatility, particularly a drop following warnings about potential supply constraints. Analysts remain divided on future projections, with target prices ranging from $355 to $400. This report explores the latest news surrounding AAPL, its financial fundamentals, and outlook for investors.

## Latest News and Developments
### Market Performance
As of early October 2026, AAPL stock is trading around $333.63, showing a year-to-date return of approximately 23.06%, significantly outperforming the S&P 500’s 14.22% gain. The stock is currently experiencing some adjustments in price target forecasts, notably from Morgan Stanley, who has reduced their target from $360 to $355. 

### Quarterly Earnings Report
In Q3 FY2026, Apple reported:  
- **Revenue**: $109.4 billion, featuring a 16% increase YoY. This surpassed analysts' expectations (projected at $108.7 billion).  
- **Diluted EPS**: Rose by 29% to $2.02, exceeding consensus estimates of $1.89, aided by a $0.11 per share tariff refund.  
- **iPhone Revenues**: Increased by 22% to $54.25 billion.  
- **Mac Sales**: Climbed 29% to $10.35 billion, achieving record June-quarter figures.  
- **Services Segment**: Grew 12% to $30.74 billion but missed forecasts due to foreign exchange challenges.  
- **Net Income**: Reached $29.8 billion, up 27% YoY.

Despite these strong figures, Apple has warned about upcoming supply constraints, which led to a significant dip of over 6% in stock value during after-hours trading following the earnings report. 

### Upcoming Announcements
Looking ahead, anticipation for Apple's Q4 earnings report set for October 28 is high, with investors keen to understand how the integration of AI in their products may influence performance. Additionally, a forthcoming announcement regarding new smart-home products on October 13 may also shape investor sentiment positively.

## Fundamentals Analysis
### Financial Metrics  
- **Market Capitalization**: $4.82 trillion  
- **EPS**: $8.72  
- **P/E Ratio**: 38.24  

### Growth Outlook
Analysts have mixed predictions, with:
- **Morgan Stanley**: Target price of $355 (upside of 7.55%).  
- **TD Cowen**: More optimistic with a target of $400 (potential gain of around 25.13%). 

### Competitive Landscape
Challenges in the AI sector have been noted, as competition from companies like Google prompts concerns over Apple's innovation rate. While AAPL continues to capitalize on its iPhone and Mac sales, its services division’s slower growth has raised flags among analysts.

## Conclusion
Apple's performance in Q3 FY2026 demonstrates strong financial health, buoyed by robust product sales. Nevertheless, the stock’s volatility connected to supply constraints must be monitored closely by investors. As the company prepares for significant product announcements and earnings reports, strategic insights into its approach towards AI integration will be crucial for assessing future performance.

## Follow-Up Research Questions
1. **How will supply constraints affect future earnings and growth forecasts for Apple?**  
2. **What specific plans does Apple have to enhance its competitive positioning in the AI sector?**  
3. **How might macroeconomic factors like inflation impact Apple’s pricing strategies and consumer demand for products?**  
4. **What role will the new smart-home product line play in Apple's revenue diversification?**  
5. **Are there any potential regulatory challenges that could impact Apple’s operational strategies in the near future?**  

This report aims to provide insights into Apple’s recent performance, its current financial health, and the competitive landscape affecting its stock, reinforcing the necessity for ongoing analysis and attention from the investment community.



------------------



## **Practice Opportunity**
### Create a new AI agent named SentimentAgent that specializes in searching online to find out the sentiment regarding a company:
#### -Update the agent's instructions so that it focuses on analyzing online sentiment (positive, negative, neutral) about a company.
#### -Use the SentimentAgent agent as a tool for the writer agent we defined earlier.
#### -Provide an example input (e.g, "What is the current market sentiment about Tesla?")
#### -Run the agent, analyze the response, and review the traces on OpenAI API platform.

In [ ]:
!pip install -q --upgrade openai-agents==0.2.2
!pip install -q tavily-python

import os, requests
from pydantic import BaseModel
from agents import Agent, SQLiteSession, RunResult, handoff, function_tool
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from typing_extensions import TypedDict

# Re-initialize basic parameters
main_model = "gpt-4o-mini"
session = SQLiteSession("research_agent_handoff_practice_2")

# Define Pydantic structures
class Summary(BaseModel):
    summary: str

class FinalReport(BaseModel):
    short_summary: str
    markdown_report: str
    follow_up_questions: list[str]

# Define Tavily search parameter model and search tool locally
class TavilyParams(TypedDict):
    query: str
    max_results: int

@function_tool
def tavily_search(params: TavilyParams) -> str:
    """Return a newline-joined summary of Tavily results."""
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": os.environ.get("TAVILY_API_KEY", ""),
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }
    resp = requests.post(url, json=payload, headers={"Content-Type": "application/json"})
    if resp.status_code != 200:
        return f"Tavily error {resp.status_code}"
    items = resp.json().get("results", [])
    return "\n".join([f".{itm['title']}: {itm['content']}" for itm in items]) or "No hits"

# Re-define base specialist agents that the writer relies on
search_agent = Agent(
    name="Searcher",
    instructions="""
Context: You are a search specialist agent with access to the Tavily web search tool.
Your goal is to provide up-to-date, relevant information for a research task.
""",
    tools=[tavily_search],
    model=main_model,
    output_type=Summary
)

fundamentals_agent = Agent(
    name="FundamentalAnalyst",
    instructions="""
Context: You are a financial analyst specializing in company fundamentals.
""",
    tools=[tavily_search],
    model=main_model,
    output_type=Summary
)

SENTIMENT_PROMPT = """
Context: You are a sentiment analyst specializing in evaluating online sentiment about companies.

Instruction: Carefully analyze the provided notes and search online sources to determine the current sentiment (positive, negative, or neutral).

Input: Notes containing relevant information and search results about the company.

Output: A concise summary (<=200 words) highlighting the overall sentiment, supporting evidence, and any notable trends or shift.

Tools: The following tools are available for comprehensive sentiment research on the company:
-tavily_search: Search the web for information about the company.
"""

sentiment_agent = Agent(
    name="SentimentAnalyst",
    instructions=SENTIMENT_PROMPT,
    output_type=Summary,
    model=main_model,
    tools=[tavily_search]
)

# Define an async extractor function
async def extract_summary(run_result: RunResult) -> str:
    """Extracts the 'summary' field from the final_output of an agent run."""
    return run_result.final_output.summary

# Define the writer agent with correct argument passing and nested tools
writer_agent = Agent(
    name="Writer",
    instructions="""
Context: You are an expert research writer preparing a comprehensive investment report on a company, using diverse information sources.
Your audience is sophisticated and expects clarity, depth, and actionable insights.

Instruction: Thoroughly analyze the provided search snippets and analyst summaries. Synthesize these into a cohesive, well-structured
markdown report of at least 600 words.
""",
    model=main_model,
    output_type=FinalReport,
    tools=[
        fundamentals_agent.as_tool(
            "fundamentals",
            "Get fundamental analysis",
            custom_output_extractor=extract_summary
        ),
        search_agent.as_tool(
            "search",
            "Get search results",
            custom_output_extractor=extract_summary
        ),
        sentiment_agent.as_tool(
            "sentiment",
            "Get sentiment analysis",
            custom_output_extractor=extract_summary
        )
    ]
)

## **Let's run the results**

In [ ]:
import os
from google.colab import userdata

# Restore API Keys to os.environ in case of kernel reset
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

from IPython.display import display, Markdown
from pydantic import BaseModel
from agents import Runner, SQLiteSession, Agent, handoff, InputGuardrail, GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from datetime import datetime

# 1. Ensure database session is established
session = SQLiteSession("research_agent_handoff_practice_2")

# 2. Define print utility function
def print_markdown(txt: str):
    display(Markdown(txt))

# 3. Define and instantiate political guardrail dependencies
class PoliticalTopicOutput(BaseModel):
    is_political: bool
    reasoning: str

politics_guardrail_agent = Agent(
    name="Guardrail Check",
    instructions="Check if the user is asking about political topics, politicians, elections, government, and so on",
    output_type=PoliticalTopicOutput
)

async def politics_guardrail_fn(ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    user_message = input if isinstance(input, str) else " ".join([item.get("text", "") for item in input if "text" in item])
    result = await Runner.run(politics_guardrail_agent, input=user_message)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_political
    )

politics_guardrail = InputGuardrail(politics_guardrail_fn)

# 4. Define schemas for handoff structure
class SearchPlanItem(BaseModel):
    reason: str
    query: str

class SearchPlan(BaseModel):
    searches: list[SearchPlanItem]

class PlannerToWriterInput(BaseModel):
    original_query: str
    search_plan: SearchPlan

# 5. Define callback & handoff interface
def on_planner_to_writer(ctx, input_data: PlannerToWriterInput):
    print_markdown("### Transfer: Planner -> Writer Agent")

handoff_to_writer = handoff(
    agent=writer_agent,
    input_type=PlannerToWriterInput,
    on_handoff=on_planner_to_writer,
    tool_name_override="transfer_to_writer",
    tool_description_override="Transfer to writer with original query and search plan"
)

# 6. Initialize Planner Agent with the newly defined politics guardrail
date = datetime.now().strftime("%Y-%m-%d")
planner_agent = Agent(
    name="Planner",
    instructions=f"""Current date:{date}\n Context: You are a research planner agent tasked with designing a comprehensive\n research plan for a user request. You have access to web search tools.\n Instruction: Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query.\n Input: The user's research request and the current date.\n Output: A list of search plan items, formatted as a JSON object matching the SearchPlan schema.\n""",
    model="gpt-4o-mini",
    output_type=SearchPlan,
    input_guardrails=[politics_guardrail]
)

# 7. Assemble Planner with Handoff capability
planner_with_handoff = planner_agent.clone(
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}\n\n"""
    + planner_agent.instructions
    + "\nwhen you have produced the SearchPlan, call the handoff tool handoff_to_writer with this JSON input: {{original_query:<the user query>, search_plan: <the SearchPlan JSON}}",
    handoffs=[handoff_to_writer]
)

# 8. Define main execution loop
async def run_handoffs_demo(user_query: str):
    print_markdown(f"## User Query\n{user_query}")
    run_res = await Runner.run(planner_with_handoff, user_query, session=session)
    print_markdown("---")

    report = run_res.final_output
    print_markdown("---")
    print_markdown(f"### Executive Summary\n{report.short_summary}")
    print_markdown("\n\n------------------\n\n")
    print_markdown(f"### Full Report\n{report.markdown_report}")
    print_markdown("\n\n------------------\n\n")
    return run_res

# 9. Execute demo run
handoff_result = await run_handoffs_demo("Do a deep dive on the latest news and developments in the AAPL stock. Also i need the fundamentals analysis of the company")

## User Query
Do a deep dive on the latest news and developments in the AAPL stock. Also i need the fundamentals analysis of the company

### Transfer: Planner -> Writer Agent

---

---

### Executive Summary
AAPL stock shows strong fundamentals and recent performance, with strategic developments in smart home tech. However, competitive challenges and valuation ratios suggest cautious optimism.



------------------



### Full Report
# A Comprehensive Investment Report on Apple Inc. (AAPL)

## 1. Overview
As of October 7, 2026, Apple Inc. (AAPL) continues to be a cornerstone in the technology sector, recognized for its influential product lines including the iPhone, Mac, and services. The stock is currently trading around **$333.63** with a market capitalization of approximately **$4.892 trillion**. Apple's recent price action has shown volatility, opening at **$337.02** and touching lows of **$332.79** and highs of **$338.67** within the current trading session.

## 2. Recent Developments
### Product Launches and Strategic Shifts
On October 13, 2026, Apple aims to announce new advancements in smart-home technologies, showcasing a strategic focus under new CEO John Ternus. This shift marks a pivotal moment for the company, signaling its intent to diversify revenue streams beyond its mobile products. Analysts expect that, soon after the announcement, the market could respond positively due to potential for new revenue growth trajectories.

### Financial Performance
Apple’s Q3 fiscal results reported revenues of **$109.4 billion**, a **16% year-over-year increase**, primarily driven by substantial iPhone and Mac sales. For the fourth quarter of FY2026, analysts project an EPS growth of **7%** from the previous year, estimating it to reach **$1.98**, contributing to a full-year EPS estimate of **$8.74**. Moreover, the forward EPS for FY2027 is estimated to reach **$9.53**, signifying continued health in earnings.

## 3. Analyst Ratings
Morgan Stanley has maintained an 'Overweight' rating on AAPL stock but reduced the price target from **$360** to **$355**. The consensus rating for AAPL remains a **Moderate Buy** among analysts, with predictions averaging a price target of about **$289.61**, suggesting a 12% upside potential. Notably, despite the generally positive sentiment, there are warnings regarding current valuation levels given the high P/E ratios.

## 4. Market Performance
### Year-to-Date Returns
AAPL has delivered **23.64%** YTD returns, significantly outperforming the S&P 500's **13.49%**. This indicates robust investor confidence despite competitive pressures, particularly from advancements in AI technologies led by companies such as Google. Nevertheless, year-to-date stock performance shows a slight decline of **5%**, reflecting competitive challenges and investor caution.

### Valuation Metrics
Apple's trailing P/E stands at **38.66**, and price/book value at **45.23**, indicating that the stock is priced at a premium compared to its earnings and net assets. The stock also shows high valuations on other indicators such as Price/Sales (10.54) and Price/Cash Flow (31.67). While such metrics suggest strong growth expectations, they also raise concerns regarding valuation sustainability if growth decelerates.

## 5. Future Challenges and Market Trends
Apple faces significant challenges ahead, especially regarding its heavy reliance on iPhone sales and increased competition within various tech sectors, particularly those leveraging AI. Rising production costs, especially in components like memory chips, add another layer of complexity to its financial outlook. As competition escalates, the potential for margin compression increases, impacting future profitability.

Additionally, the modest dividend yield of **0.38%** may not attract growth-focused investors seeking returns, putting pressure on shares to perform consistently.

## 6. Conclusion
In summary, while AAPL maintains robust financial fundamentals and is positioned for growth through strategic product developments, key challenges exist that may limit short-term upside potential. Investors should remain cautious, monitoring upcoming product launches and market landscape changes closely, particularly how they affect earnings and overall stock performance.

### Actionable Insights
- **Monitor Upcoming Product Announcements**: The smart home technology launch could signal new revenue opportunities.
- **Track Competitive Developments**: Keep an eye on competitor advancements in AI and technology to gauge potential impacts on Apple's market share and innovation trajectory.
- **Evaluate Earnings Calls**: The insights from the upcoming earnings call on October 28 could provide clarifying information concerning growth outlook and financial health, pivotal for investment decisions.



------------------

